In [1]:
# load the model for blimp dynamics 

import pickle as pkl
import torch
from saviolo_et_al_mlp import DiscreteQuadDynamicsNN


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sav_model = DiscreteQuadDynamicsNN()
sav_model.to(device)
print(device)

sav_model.load_model("trained_models/trained_model_noisy_spiral.pth")

cuda


c:\Users\ksubh\OneDrive\Documents\SwarmsLab_RL\agile-blimp\mochi-swarm-sim\saviolo_et_al_mlp.py:204: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch

In [ ]:
# MPPI (Model Predictive Path Integral) for Blimp Position Control
import numpy as np
import torch
import torch.nn.functional as F

class BlimpMPPI:
    """
    MPPI controller for blimp position control.
    The dynamics model outputs velocity, which is integrated to get position.
    """
    def __init__(
        self,
        dynamics_model,
        horizon: int = 20,
        num_samples: int = 1000,
        lambda_: float = 1.0,
        sigma: float = 0.5,
        dt: float = 0.05,
        device: str = "cuda"
    ):
        """
        Args:
            dynamics_model: The blimp dynamics model (sav_model) with .sample() method
            horizon: Prediction horizon (number of steps)
            num_samples: Number of action sequence samples
            lambda_: Temperature parameter for importance weighting
            sigma: Standard deviation for action noise
            dt: Time step for integration
            device: Device to run computations on
        """
        self.model = dynamics_model
        self.horizon = horizon
        self.num_samples = num_samples
        self.lambda_ = lambda_
        self.sigma = sigma
        self.dt = dt
        self.device = device
        
        # Action bounds (adjust based on your action space)
        self.action_min = torch.tensor([-1.0, -1.0, -1.0], device=device)
        self.action_max = torch.tensor([1.0, 1.0, 1.0], device=device)
        
        # Initialize action sequence (will be updated each iteration)
        self.action_sequence = torch.zeros((horizon, 3), device=device)
        
    def sample_action_sequences(self, mean_actions: torch.Tensor) -> torch.Tensor:
        """
        Sample action sequences from Gaussian distribution around mean.
        
        Args:
            mean_actions: Mean action sequence (horizon, 3)
            
        Returns:
            Sampled action sequences (num_samples, horizon, 3)
        """
        noise = torch.randn(
            self.num_samples, self.horizon, 3, 
            device=self.device
        ) * self.sigma
        
        actions = mean_actions.unsqueeze(0) + noise  # (num_samples, horizon, 3)
        
        # Clip to action bounds
        actions = torch.clamp(actions, self.action_min, self.action_max)
        
        return actions
    
    def rollout_trajectories(
        self, 
        initial_state: torch.Tensor,
        initial_position: torch.Tensor,
        action_sequences: torch.Tensor
    ) -> tuple:
        """
        Roll out trajectories using the dynamics model.
        Integrates velocity to get position.
        Uses batched forward pass for efficiency.
        
        Args:
            initial_state: Initial state [v(3), w(3), rpy(3)] shape (9,)
            initial_position: Initial position [x, y, z] shape (3,)
            action_sequences: Action sequences (num_samples, horizon, 3)
            
        Returns:
            positions: (num_samples, horizon+1, 3) - includes initial position
            states: (num_samples, horizon+1, 9) - includes initial state
        """
        num_samples = action_sequences.shape[0]
        
        # Initialize trajectories
        positions = torch.zeros(
            (num_samples, self.horizon + 1, 3), 
            device=self.device
        )
        positions[:, 0] = initial_position.unsqueeze(0).repeat(num_samples, 1)
        
        states = torch.zeros(
            (num_samples, self.horizon + 1, 9), 
            device=self.device
        )
        states[:, 0] = initial_state.unsqueeze(0).repeat(num_samples, 1)
        
        # Initialize current observations for all samples (batched)
        current_obs = initial_state.unsqueeze(0).repeat(num_samples, 1)  # (num_samples, 9)
        
        # Rollout each step
        for t in range(self.horizon):
            # Get actions for this timestep (num_samples, 3)
            actions = action_sequences[:, t, :]  # (num_samples, 3)
            
            # Prepare batched input: [state, action] for all samples
            model_input = torch.cat([current_obs, actions], dim=-1)  # (num_samples, 12)
            model_input_norm = (model_input - self.model.x_mean) / self.model.x_std
            
            # Forward pass (batched) - much faster than sequential
            with torch.no_grad():
                next_obs_norm = self.model.sample(model_input_norm)  # (num_samples, 9)
                next_obs = next_obs_norm * self.model.y_std + self.model.y_mean
            
            states[:, t+1] = next_obs
            current_obs = next_obs  # Update for next iteration
            
            # Extract velocity (first 3 elements of state)
            velocities = next_obs[:, 0:3]  # (num_samples, 3)
            
            # Integrate velocity to get position
            positions[:, t+1] = positions[:, t] + velocities * self.dt
        
        return positions, states
    
    def compute_costs(
        self,
        positions: torch.Tensor,
        target_position: torch.Tensor,
        states: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Compute cost for each trajectory.
        
        Args:
            positions: (num_samples, horizon+1, 3)
            target_position: (3,) target position [x, y, z]
            states: (num_samples, horizon+1, 9) optional, for additional costs
            
        Returns:
            costs: (num_samples,) total cost per trajectory
        """
        target = target_position.unsqueeze(0).unsqueeze(0)  # (1, 1, 3)
        
        # Position error cost (MSE over horizon)
        position_errors = positions - target  # (num_samples, horizon+1, 3)
        position_costs = torch.sum(position_errors ** 2, dim=(1, 2))  # (num_samples,)
        
        # Optional: Add control cost (penalize large actions)
        # This would require storing actions during rollout
        
        # Optional: Add state-based costs (e.g., orientation penalties)
        # if states is not None:
        #     orientation_costs = ...
        
        return position_costs
    
    def update_action_sequence(
        self,
        action_sequences: torch.Tensor,
        costs: torch.Tensor
    ) -> torch.Tensor:
        """
        Update action sequence using importance weighting.
        
        Args:
            action_sequences: (num_samples, horizon, 3)
            costs: (num_samples,)
            
        Returns:
            Updated mean action sequence (horizon, 3)
        """
        # Compute importance weights
        # w_i = exp(-(1/lambda) * (S_i - min(S)))
        min_cost = torch.min(costs)
        weights = torch.exp(-(1.0 / self.lambda_) * (costs - min_cost))
        weights = weights / (torch.sum(weights) + 1e-8)  # Normalize
        
        # Weighted average of action sequences
        weights_expanded = weights.unsqueeze(-1).unsqueeze(-1)  # (num_samples, 1, 1)
        updated_sequence = torch.sum(
            action_sequences * weights_expanded, 
            dim=0
        )  # (horizon, 3)
        
        return updated_sequence
    
    def compute_action(
        self,
        current_state: torch.Tensor,
        current_position: torch.Tensor,
        target_position: torch.Tensor
    ) -> torch.Tensor:
        """
        Compute optimal action using MPPI.
        
        Args:
            current_state: Current state [v(3), w(3), rpy(3)] shape (9,)
            current_position: Current position [x, y, z] shape (3,)
            target_position: Target position [x, y, z] shape (3,)
            
        Returns:
            Optimal action (3,)
        """
        # Ensure tensors are on correct device
        if isinstance(current_state, np.ndarray):
            current_state = torch.from_numpy(current_state).float().to(self.device)
        if isinstance(current_position, np.ndarray):
            current_position = torch.from_numpy(current_position).float().to(self.device)
        if isinstance(target_position, np.ndarray):
            target_position = torch.from_numpy(target_position).float().to(self.device)
        
        # Sample action sequences around current mean
        action_sequences = self.sample_action_sequences(self.action_sequence)
        
        # Rollout trajectories
        positions, states = self.rollout_trajectories(
            current_state, current_position, action_sequences
        )
        
        # Compute costs
        costs = self.compute_costs(positions, target_position, states)
        
        # Update action sequence
        self.action_sequence = self.update_action_sequence(action_sequences, costs)
        
        # Return first action from updated sequence
        return self.action_sequence[0].clone()
    
    def reset(self):
        """Reset the action sequence."""
        self.action_sequence = torch.zeros((self.horizon, 3), device=self.device)

In [ ]:
# Example usage of MPPI controller

# Initialize MPPI controller
mppi = BlimpMPPI(
    dynamics_model=sav_model,
    horizon=20,           # Prediction horizon
    num_samples=500,      # Number of samples (reduce for faster computation)
    lambda_=1.0,          # Temperature parameter
    sigma=0.5,            # Action noise standard deviation
    dt=0.05,              # Time step (should match your simulation dt)
    device=device
)

# Example: Compute action for position control
# Assume you have:
# - current_state: [vx, vy, vz, wx, wy, wz, roll, pitch, yaw] (9D)
# - current_position: [x, y, z] (3D)
# - target_position: [x_target, y_target, z_target] (3D)

# Example values (replace with actual state/position from your system)
current_state = torch.zeros(9, device=device)  # [v(3), w(3), rpy(3)]
current_position = torch.tensor([0.0, 0.0, 1.0], device=device)  # [x, y, z]
target_position = torch.tensor([5.0, 2.0, 2.0], device=device)  # [x_target, y_target, z_target]

# Compute optimal action
optimal_action = mppi.compute_action(
    current_state=current_state,
    current_position=current_position,
    target_position=target_position
)

print(f"Optimal action: {optimal_action.cpu().numpy()}")
print(f"Action shape: {optimal_action.shape}")


In [ ]:
# Note: The base BlimpMPPI class now uses batched rollouts by default for efficiency.
# The rollout function directly uses model.forward() with batched inputs, which is
# much faster than calling model.sample() sequentially for each sample.


In [ ]:
# Example control loop for position tracking with MuJoCo simulator
# This shows how to use MPPI with the actual simulator instead of the NN model

import mujoco as mj
from src.simulation import Simulation
from src.controller import Controller
from src.definitions import THRUST_LEFT, THRUST_RIGHT, SERVO, IMU_POS, IMU_LIN_VEL, IMU_ANG_VEL, IMU_QUAT, State
from scipy.spatial.transform import Rotation as R

class MPPIController(Controller):
    """
    Custom controller that accepts MPPI actions directly.
    """
    def __init__(self, model, data, mppi_action=None):
        super().__init__(model, data)
        self.mppi_action = mppi_action  # Will be set externally
        
    def control_step(self, model, data):
        """
        Override control_step to use MPPI actions directly.
        """
        if self.mppi_action is not None:
            # MPPI actions are in range [-1, 1] for all dimensions
            # Map to simulator actuator ranges:
            # - left_thrust, right_thrust: [-1, 1] -> [0, 1]
            # - servo_angle: [-1, 1] -> [-π, π]
            left_thrust = (self.mppi_action[0] + 1.0) / 2.0  # [-1,1] -> [0,1]
            right_thrust = (self.mppi_action[1] + 1.0) / 2.0  # [-1,1] -> [0,1]
            servo_angle = self.mppi_action[2] * np.pi  # [-1,1] -> [-π, π]
            
            # Clamp to valid ranges
            left_thrust = np.clip(left_thrust, 0.0, 1.0)
            right_thrust = np.clip(right_thrust, 0.0, 1.0)
            servo_angle = np.clip(servo_angle, -np.pi, np.pi)
            
            # Apply actuator commands directly
            data.actuator(THRUST_LEFT).ctrl = left_thrust
            data.actuator(THRUST_RIGHT).ctrl = right_thrust
            data.actuator(SERVO).ctrl = servo_angle
        else:
            # Fallback: no action (motors off)
            data.actuator(THRUST_LEFT).ctrl = 0.0
            data.actuator(THRUST_RIGHT).ctrl = 0.0
            data.actuator(SERVO).ctrl = np.pi / 2.0

def extract_state_from_simulator(controller):
    """
    Extract state vector from simulator sensors.
    Returns: [vx, vy, vz, wx, wy, wz, roll, pitch, yaw] (9D)
    """
    # Get linear velocity [vx, vy, vz]
    lin_vel = controller.data.sensor(IMU_LIN_VEL).data.copy()
    vx, vy, vz = lin_vel[0], lin_vel[1], lin_vel[2]
    
    # Get angular velocity [wx, wy, wz]
    ang_vel = controller.data.sensor(IMU_ANG_VEL).data.copy()
    wx, wy, wz = ang_vel[0], ang_vel[1], ang_vel[2]
    
    # Get orientation (quaternion -> euler)
    quat = controller.data.sensor(IMU_QUAT).data.copy()  # [w, x, y, z]
    r = R.from_quat([quat[1], quat[2], quat[3], quat[0]])  # scipy uses [x, y, z, w]
    roll, pitch, yaw = r.as_euler("xyz", degrees=False)  # in radians
    
    return np.array([vx, vy, vz, wx, wy, wz, roll, pitch, yaw])

def extract_position_from_simulator(controller):
    """
    Extract position [x, y, z] from simulator.
    """
    pos = controller.data.sensor(IMU_POS).data.copy()
    return pos.copy()  # [x, y, z]

def run_mppi_control_loop_with_simulator(
    mppi_controller,
    initial_position,
    target_position,
    num_steps=100,
    dt=0.05,
    model_xml_path="models/mochi.xml"
):
    """
    Run a control loop using MPPI with the actual MuJoCo simulator.
    
    Args:
        mppi_controller: BlimpMPPI instance
        initial_position: Initial position [x, y, z] (numpy array or torch tensor)
        target_position: Target position [x, y, z] (numpy array or torch tensor)
        num_steps: Number of control steps
        dt: Time step
        model_xml_path: Path to MuJoCo XML model
        
    Returns:
        positions: List of positions over time
        states: List of states over time
        actions: List of actions over time
    """
    # Load MuJoCo model
    model = mj.MjModel.from_xml_path(model_xml_path)
    data = mj.MjData(model)
    
    # Create custom MPPI controller
    controller = MPPIController(model, data)
    
    # Set up simulation (headless mode for speed)
    sim = Simulation(model, data, controller, headless=True)
    
    # Reset simulation to initial state
    mj.mj_resetData(model, data)
    mj.mj_forward(model, data)
    
    # Convert target position to torch tensor
    if isinstance(target_position, np.ndarray):
        target_pos = torch.from_numpy(target_position).float().to(mppi_controller.device)
    else:
        target_pos = target_position.clone()
    
    # Storage
    positions = []
    states = []
    actions = []
    
    print("Starting MPPI control loop with simulator...")
    
    # Extract and store initial state/position
    initial_state_np = extract_state_from_simulator(controller)
    initial_position_np = extract_position_from_simulator(controller)
    positions.append(initial_position_np.copy())
    states.append(initial_state_np.copy())
    
    for step in range(num_steps):
        # Extract current state and position from simulator
        current_state_np = extract_state_from_simulator(controller)
        current_position_np = extract_position_from_simulator(controller)
        
        # Convert to torch tensors
        current_state = torch.from_numpy(current_state_np).float().to(mppi_controller.device)
        current_position = torch.from_numpy(current_position_np).float().to(mppi_controller.device)
        
        # Compute action using MPPI
        action = mppi_controller.compute_action(
            current_state=current_state,
            current_position=current_position,
            target_position=target_pos
        )
        action_np = action.cpu().numpy()
        actions.append(action_np.copy())
        
        # Set action in controller
        controller.mppi_action = action_np
        
        # Run one simulation step (this will call control_step which applies the action)
        mj.mj_step(model, data)
        
        # Extract and store state/position after step
        next_state_np = extract_state_from_simulator(controller)
        next_position_np = extract_position_from_simulator(controller)
        positions.append(next_position_np.copy())
        states.append(next_state_np.copy())
        
        # Compute error
        error = np.linalg.norm(next_position_np - target_position)
        if step % 10 == 0:
            print(f"Step {step}: Position error = {error:.3f}, Position = {next_position_np}")
    
    # Clean up
    mj.set_mjcb_control(None)
    
    return positions, states, actions

# Example usage (uncomment to run):
# positions, states, actions = run_mppi_control_loop_with_simulator(
#     mppi_controller=mppi,
#     initial_position=np.array([0.0, 0.0, 1.0]),
#     target_position=np.array([5.0, 2.0, 2.0]),
#     num_steps=200,
#     dt=0.05
# )
